# SP-8502: Sesión 9 - Regresión Lineal
## VERSIÓN LIMPIA (Sin importaciones problemáticas)
Laboratorio Autoguiado - Pesquerías Artesanales

In [ ]:
# INSTALACIÓN Y IMPORTS - SOLO LO NECESARIO
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import shapiro, levene, norm
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.outliers_influence import variance_inflation_factor as VIF
from statsmodels.graphics.gofplots import ProbPlot
from statsmodels.stats.stattools import durbin_watson
import warnings
warnings.filterwarnings('ignore')

# Configuración
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

print("\n" + "="*80)
print("LABORATORIO AUTOGUIADO: REGRESIÓN LINEAL - SESIÓN 9")
print("SP-8502: Métodos Cuanti-Cuali con IA Responsable")
print("="*80)

## PARTE 1: Generación de Dataset

In [ ]:
print("\n[PARTE 1] Generación del Dataset Sintético")
print("-" * 80)

np.random.seed(42)
n = 150

data = {
    'id': np.arange(1, n + 1),
    'horas_faena': np.random.normal(6.5, 1.2, n),
    'tamanio_emb': np.random.normal(7.2, 2.1, n),
    'experiencia': np.random.uniform(1, 45, n),
    'temperatura': np.random.normal(24.5, 1.8, n),
}

# Generar Y con proceso conocido
beta_0, beta_1, beta_2, beta_3 = 15.0, 8.5, 3.2, 0.4
epsilon = np.random.normal(0, 5.0, n)
data['captura'] = (beta_0 + beta_1 * data['horas_faena'] + 
                   beta_2 * data['tamanio_emb'] + beta_3 * data['experiencia'] + epsilon)
data['captura'] = np.maximum(data['captura'], 5)

df = pd.DataFrame(data)
print(f"✓ {df.shape[0]} observaciones, {df.shape[1]} variables")
print(f"\nEstadísticas:\n{df[['captura', 'horas_faena', 'tamanio_emb', 'experiencia']].describe().round(3)}")

## PARTE 2: Correlaciones

In [ ]:
print("\n[PARTE 2] Matriz de Correlaciones")
print("-" * 80)

cols = ['captura', 'horas_faena', 'tamanio_emb', 'experiencia', 'temperatura']
corr = df[cols].corr()
print(corr.round(3))

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.3f', cmap='coolwarm', center=0, 
            square=True, linewidths=1, ax=ax, vmin=-1, vmax=1)
ax.set_title('Matriz de Correlaciones\nPesquerías Artesanales', 
             fontsize=12, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

## PARTE 3: Regresión Lineal Simple

In [ ]:
print("\n[PARTE 3] Regresión Lineal Simple")
print("-" * 80)

m_simple = ols('captura ~ horas_faena', data=df).fit()
print(m_simple.summary())

fig, ax = plt.subplots(figsize=(11, 6))
ax.scatter(df['horas_faena'], df['captura'], alpha=0.6, s=70, 
          color='steelblue', edgecolors='black', linewidth=0.5)
x_pred = np.linspace(df['horas_faena'].min(), df['horas_faena'].max(), 100)
y_pred = m_simple.predict(pd.DataFrame({'horas_faena': x_pred}))
ax.plot(x_pred, y_pred, 'r-', linewidth=2.5, 
        label=f"R² = {m_simple.rsquared:.3f}")
ax.set_xlabel('Horas de Faena', fontsize=11, fontweight='bold')
ax.set_ylabel('Captura (kg)', fontsize=11, fontweight='bold')
ax.set_title('Regresión Lineal Simple', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## PARTE 4: Regresión Lineal Múltiple

In [ ]:
print("\n[PARTE 4] Regresión Lineal Múltiple")
print("-" * 80)

m_mult = ols('captura ~ horas_faena + tamanio_emb + experiencia', data=df).fit()
print(m_mult.summary())

## PARTE 5: DIAGNÓSTICOS DE SUPUESTOS

In [ ]:
print("\n[PARTE 5] DIAGNÓSTICOS DE SUPUESTOS")
print("-" * 80)

resid = m_mult.resid
fitted = m_mult.fittedvalues

# 5.1: LINEALIDAD
print("\n[5.1] Linealidad")
fig, ax = plt.subplots(figsize=(11, 6))
ax.scatter(fitted, resid, alpha=0.6, s=70, color='steelblue', edgecolors='black', linewidth=0.5)
ax.axhline(y=0, color='r', linestyle='--', linewidth=2)
ax.set_xlabel('Valores Ajustados (ŷ)', fontsize=11, fontweight='bold')
ax.set_ylabel('Residuales (e)', fontsize=11, fontweight='bold')
ax.set_title('Diagnóstico 1: Residuales vs. Valores Ajustados', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print("✓ Residuales sin patrón claro = Linealidad OK")

In [ ]:
# 5.2: NORMALIDAD
print("\n[5.2] Normalidad de Residuales")
stat_sw, pval_sw = shapiro(resid)
fig, axes = plt.subplots(2, 2, figsize=(13, 11))

# Histograma
axes[0, 0].hist(resid, bins=20, color='steelblue', edgecolor='black', alpha=0.7, density=True)
mu, sigma = resid.mean(), resid.std()
x = np.linspace(resid.min(), resid.max(), 100)
axes[0, 0].plot(x, stats.norm.pdf(x, mu, sigma), 'r-', linewidth=2)
axes[0, 0].set_title('(A) Histograma de Residuales', fontsize=11, fontweight='bold')
axes[0, 0].set_xlabel('Residuales', fontsize=10)
axes[0, 0].grid(True, alpha=0.3)

# Q-Q Plot
pp = ProbPlot(resid)
pp.qqplot(ax=axes[0, 1], line='45', alpha=0.6, markersize=7)
axes[0, 1].set_title('(B) Q-Q Plot', fontsize=11, fontweight='bold')
axes[0, 1].grid(True, alpha=0.3)

# Test Shapiro-Wilk
axes[1, 0].text(0.5, 0.6, f'Shapiro-Wilk Test\nW = {stat_sw:.4f}\np = {pval_sw:.4f}', 
               ha='center', fontsize=12, transform=axes[1, 0].transAxes,
               bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.7))
axes[1, 0].text(0.5, 0.2, ('✓ Normales (p ≥ 0.05)' if pval_sw >= 0.05 else '✗ NO Normales (p < 0.05)'),
               ha='center', fontsize=11, transform=axes[1, 0].transAxes, fontweight='bold')
axes[1, 0].axis('off')

# ECDF
sorted_resid = np.sort(resid)
ecdf = np.arange(1, len(sorted_resid) + 1) / len(sorted_resid)
normal_ecdf = norm.cdf(sorted_resid, mu, sigma)
axes[1, 1].plot(sorted_resid, ecdf, 'b-', linewidth=2, label='ECDF empírica')
axes[1, 1].plot(sorted_resid, normal_ecdf, 'r--', linewidth=2, label='CDF Normal teórica')
axes[1, 1].set_title('(D) Función Acumulada', fontsize=11, fontweight='bold')
axes[1, 1].legend(fontsize=9)
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print(f"Shapiro-Wilk: p = {pval_sw:.4f} → {'✓ Normales' if pval_sw >= 0.05 else '✗ NO Normales'}")

In [ ]:
# 5.3: HOMOCEDASTICIDAD
print("\n[5.3] Homocedasticidad (Varianza Constante)")
std_resid = resid / resid.std()
median_fitted = fitted.median()
g1 = resid[fitted < median_fitted]
g2 = resid[fitted >= median_fitted]
stat_lev, pval_lev = levene(g1, g2)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scale-Location
axes[0].scatter(fitted, np.sqrt(np.abs(std_resid)), alpha=0.6, s=70, color='steelblue', edgecolors='black', linewidth=0.5)
axes[0].axhline(y=np.sqrt(2), color='r', linestyle='--', linewidth=2)
axes[0].set_xlabel('Valores Ajustados', fontsize=11, fontweight='bold')
axes[0].set_ylabel('√|Residuales Estandarizados|', fontsize=11, fontweight='bold')
axes[0].set_title('(A) Scale-Location Plot', fontsize=11, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Test Levene
axes[1].text(0.5, 0.6, f'Test de Levene\nStat = {stat_lev:.4f}\np = {pval_lev:.4f}', 
            ha='center', fontsize=12, transform=axes[1].transAxes,
            bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.7))
axes[1].text(0.5, 0.2, ('✓ Homocedástico (p ≥ 0.05)' if pval_lev >= 0.05 else '✗ Heterocedástico (p < 0.05)'),
            ha='center', fontsize=11, transform=axes[1].transAxes, fontweight='bold')
axes[1].axis('off')

plt.tight_layout()
plt.show()
print(f"Levene: p = {pval_lev:.4f} → {'✓ Homocedástico' if pval_lev >= 0.05 else '✗ Heterocedástico'}")

In [ ]:
# 5.4: INDEPENDENCIA
print("\n[5.4] Independencia de Residuales")
dw = durbin_watson(resid)
print(f"Durbin-Watson: {dw:.4f} (≈2 = OK, <2 = autocorr positiva, >2 = autocorr negativa)")

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(range(len(resid)), resid.values, marker='o', linestyle='-', alpha=0.6, 
       color='steelblue', markersize=4)
ax.axhline(y=0, color='r', linestyle='--', linewidth=2)
ax.fill_between(range(len(resid)), -2*resid.std(), 2*resid.std(), alpha=0.2, color='gray')
ax.set_xlabel('Orden de Observación', fontsize=11, fontweight='bold')
ax.set_ylabel('Residual', fontsize=11, fontweight='bold')
ax.set_title('Gráfico de Secuencia: Residuales en Orden\n(Verifica: Patrón aleatorio)',
            fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print("✓ Puntos dispersos aleatoriamente = Independencia OK")

## PARTE 6: Multicolinealidad

In [ ]:
print("\n[PARTE 6] Análisis de Multicolinealidad")
print("-" * 80)

X_num = df[['horas_faena', 'tamanio_emb', 'experiencia']]
X_const = sm.add_constant(X_num)

vif_data = pd.DataFrame()
vif_data["Variable"] = X_num.columns
vif_data["VIF"] = [VIF(X_const.values, i) for i in range(1, X_const.shape[1])]

print("\nVariance Inflation Factor (VIF):")
print(vif_data.to_string(index=False))
print("Criterio: VIF < 5 = OK, 5-10 = Revisar, >10 = Problema")

corr_x = X_num.corr()
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_x, annot=True, fmt='.3f', cmap='coolwarm', center=0, 
            square=True, linewidths=2, ax=ax, vmin=-1, vmax=1)
ax.set_title('Matriz de Correlaciones: Predictores\n(Detectar multicolinealidad)',
            fontsize=12, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

## PARTE 7: Transformaciones

In [ ]:
print("\n[PARTE 7] Transformaciones de Variables")
print("-" * 80)

print("\n[7.1] Transformación Log(Y)")
df['log_captura'] = np.log(df['captura'])
m_log = ols('log_captura ~ horas_faena + tamanio_emb + experiencia', data=df).fit()

resid_log = m_log.resid
stat_sw_log, pval_sw_log = shapiro(resid_log)

print(f"Comparación Shapiro-Wilk:")
print(f"  Original: W = {stat_sw:.4f}, p = {pval_sw:.4f}")
print(f"  log(Y):   W = {stat_sw_log:.4f}, p = {pval_sw_log:.4f}")
print(f"  Mejora: {'✓ Sí' if pval_sw_log > pval_sw else '✗ No'}")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].hist(resid, bins=20, color='steelblue', alpha=0.7, edgecolor='black', density=True)
mu1, sigma1 = resid.mean(), resid.std()
x1 = np.linspace(resid.min(), resid.max(), 100)
axes[0].plot(x1, stats.norm.pdf(x1, mu1, sigma1), 'r-', linewidth=2)
axes[0].set_title(f'Original (p={pval_sw:.4f})', fontsize=11, fontweight='bold')
axes[0].set_xlabel('Residuales')
axes[0].grid(True, alpha=0.3)

axes[1].hist(resid_log, bins=20, color='lightgreen', alpha=0.7, edgecolor='black', density=True)
mu2, sigma2 = resid_log.mean(), resid_log.std()
x2 = np.linspace(resid_log.min(), resid_log.max(), 100)
axes[1].plot(x2, stats.norm.pdf(x2, mu2, sigma2), 'r-', linewidth=2)
axes[1].set_title(f'log(Y) (p={pval_sw_log:.4f})', fontsize=11, fontweight='bold')
axes[1].set_xlabel('Residuales')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
print("\n[7.2] Estandarización de Predictores")
X_scaled = (X_num - X_num.mean()) / X_num.std()
X_scaled.columns = [c + '_sc' for c in X_scaled.columns]
df_scaled = pd.concat([df[['captura']], X_scaled], axis=1)

m_scaled = ols('captura ~ horas_faena_sc + tamanio_emb_sc + experiencia_sc', data=df_scaled).fit()

print("\nCoeficientes Estandarizados:")
for var in m_scaled.params.index[1:]:
    print(f"  {var}: {m_scaled.params[var]:.4f}")
print("→ Permite comparar importancia relativa de predictores")

## PARTE 8: Outliers e Influencia (SIN IMPORTACIONES PROBLEMÁTICAS)

In [ ]:
print("\n[PARTE 8] Detección de Outliers y Observaciones Influyentes")
print("-" * 80)

# Obtener medidas de influencia
influence = m_mult.get_influence()
cooks_d = influence.cooks_distance[0]
leverage = influence.hat_diag
dfbetas = influence.dfbetas
std_resid = resid / resid.std()

# Crear 4 gráficos sin importación de abline_plot
fig, axes = plt.subplots(2, 2, figsize=(14, 11))

# Cook's Distance
axes[0, 0].stem(range(len(cooks_d)), cooks_d, markerfmt=',', basefmt=' ')
umbral_cook = 4/len(df)
axes[0, 0].axhline(y=umbral_cook, color='r', linestyle='--', linewidth=2, label=f'Umbral: 4/n={umbral_cook:.3f}')
axes[0, 0].set_xlabel('Observación ID', fontsize=10, fontweight='bold')
axes[0, 0].set_ylabel("Cook's Distance", fontsize=10, fontweight='bold')
axes[0, 0].set_title("(A) Cook's Distance", fontsize=11, fontweight='bold')
axes[0, 0].legend(fontsize=9)
axes[0, 0].grid(True, alpha=0.3)

# Residuales Estandarizados
axes[0, 1].scatter(range(len(std_resid)), std_resid, alpha=0.6, s=60, color='steelblue', edgecolors='black', linewidth=0.5)
axes[0, 1].axhline(y=2, color='r', linestyle='--', linewidth=1.5, label='±2 SD')
axes[0, 1].axhline(y=-2, color='r', linestyle='--', linewidth=1.5)
axes[0, 1].set_xlabel('Observación ID', fontsize=10, fontweight='bold')
axes[0, 1].set_ylabel('Residual Estandarizado', fontsize=10, fontweight='bold')
axes[0, 1].set_title('(B) Residuales Estandarizados', fontsize=11, fontweight='bold')
axes[0, 1].legend(fontsize=9)
axes[0, 1].grid(True, alpha=0.3)

# Leverage
umbral_lev = 2*m_mult.df_model/len(df)
axes[1, 0].scatter(range(len(leverage)), leverage, alpha=0.6, s=60, color='orange', edgecolors='black', linewidth=0.5)
axes[1, 0].axhline(y=umbral_lev, color='r', linestyle='--', linewidth=2, label=f'Umbral: 2p/n={umbral_lev:.3f}')
axes[1, 0].set_xlabel('Observación ID', fontsize=10, fontweight='bold')
axes[1, 0].set_ylabel('Leverage', fontsize=10, fontweight='bold')
axes[1, 0].set_title('(C) Leverage (Hat Diagonal)', fontsize=11, fontweight='bold')
axes[1, 0].legend(fontsize=9)
axes[1, 0].grid(True, alpha=0.3)

# DFBETAS
for i, col in enumerate(X_num.columns[:2]):
    axes[1, 1].scatter(range(len(dfbetas)), dfbetas[:, i+1], alpha=0.6, s=50, label=col)
umbral_dfb = 2/np.sqrt(len(df))
axes[1, 1].axhline(y=umbral_dfb, color='r', linestyle='--', linewidth=1.5, label='Umbral')
axes[1, 1].axhline(y=-umbral_dfb, color='r', linestyle='--', linewidth=1.5)
axes[1, 1].set_xlabel('Observación ID', fontsize=10, fontweight='bold')
axes[1, 1].set_ylabel('DFBETAS', fontsize=10, fontweight='bold')
axes[1, 1].set_title('(D) DFBETAS', fontsize=11, fontweight='bold')
axes[1, 1].legend(fontsize=8)
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Identificar casos problemáticos
outlier_cook = np.where(cooks_d > umbral_cook)[0]
outlier_resid = np.where(np.abs(std_resid) > 2)[0]
outlier_lev = np.where(leverage > umbral_lev)[0]

print(f"\n✓ Observaciones Problemáticas:")
print(f"  Cook's D alto: {len(outlier_cook)} obs")
if len(outlier_cook) > 0:
    print(f"    IDs: {outlier_cook[:5].tolist()}{'...' if len(outlier_cook) > 5 else ''}")
print(f"  Residuales extremos (|z|>2): {len(outlier_resid)} obs")
if len(outlier_resid) > 0:
    print(f"    IDs: {outlier_resid[:5].tolist()}{'...' if len(outlier_resid) > 5 else ''}")
print(f"  Leverage alto: {len(outlier_lev)} obs")
if len(outlier_lev) > 0:
    print(f"    IDs: {outlier_lev[:5].tolist()}{'...' if len(outlier_lev) > 5 else ''}")

## PARTE 9: Resumen Comparativo

In [ ]:
print("\n[PARTE 9] Comparación de Modelos")
print("-" * 80)

modelos = {
    'Simple': m_simple,
    'Múltiple': m_mult,
    'log(Y)': m_log,
    'Escalado': m_scaled
}

comp_data = []
for name, mod in modelos.items():
    comp_data.append({
        'Modelo': name,
        'R²': f"{mod.rsquared:.4f}",
        'R² Adj': f"{mod.rsquared_adj:.4f}",
        'AIC': f"{mod.aic:.1f}",
        'BIC': f"{mod.bic:.1f}"
    })

comp_df = pd.DataFrame(comp_data)
print(f"\n{comp_df.to_string(index=False)}")
print(f"\n✓ RECOMENDADO: Modelo Múltiple")
print(f"  Razones: Balance entre ajuste (R²={m_mult.rsquared:.4f}) y complejidad")

## RESUMEN FINAL

In [ ]:
print("\n" + "="*80)
print("✅ LABORATORIO COMPLETADO")
print("="*80)
print(f"""
📊 Análisis realizados:
   ✓ Regresión lineal simple y múltiple
   ✓ 4 supuestos verificados (linealidad, normalidad, homocedasticidad, independencia)
   ✓ Diagnósticos visuales y tests estadísticos
   ✓ VIF para multicolinealidad
   ✓ Transformaciones (log, estandarización)
   ✓ Outliers e influencia (Cook's D, leverage, DFBETAS)
   ✓ Comparación de 4 modelos candidatos

📁 Objetos disponibles:
   - df: Dataset
   - m_simple, m_mult, m_log, m_scaled: Modelos ajustados

🎯 PRÓXIMOS PASOS:
   1. Adapta el código a TUS datos
   2. Sigue el flujo: Exploración → Ajuste → Diagnósticos
   3. Documenta supuestos (OK / Condicionado / Violado)
   4. Prepara tabla de resultados en formato APA
   5. Guarda gráficos (300 dpi) para entrega

💡 Para tu análisis específico, comparte:
   - Pregunta de tesis
   - Variables Y, X₁, X₂, X₃
   Y te haré un checklist personalizado
""")
print("="*80)